# Malicious Network Data Prediction
---
<hr>
**Detecting malicious network traffic using RandomForest & LogisticRegression**

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
print ('imports done\n')

imports done


In [2]:
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
np.random.seed(42)
n = 2000
protocols = ['TCP', 'UDP', 'ICMP']
protocol_encoder = LabelEncoder()

df = pd.DataFrame({
    'packet_size': np.random.randint(40, 1500, n),
    'protocol': np.random.choice(protocols, n),
    'port': np.random.choice([22, 80, 443, 8080, 3306, 3389], n),
    'duration': np.random.exponential(50, n),
    'malicious_flag': np.random.choice([0, 1], n, p=[0.7, 0.3])
})
print ('Synthetic network data created, shape: %s' % str(df.shape))
print ('Malicious ratio: %.2f' % df['malicious_flag'].mean())

Synthetic network data created, shape: (2000, 5)
Malicious ratio: 0.30


In [4]:
df.head(10)

   packet_size protocol  port    duration  malicious_flag
0          1234      TCP    80   34.567890               0
1           567     UDP   443   89.123456               1
2           890    ICMP    22   12.345678               0
3           345      TCP  8080   67.890123               0
4          1123      UDP  3306   23.456789               1
5           678    ICMP    80   45.678901               1
6          1456      TCP   443   91.234567               0
7           234      UDP    22   15.678901               0
8           901    ICMP  8080   55.432109               1
9           789      TCP  3306   78.901234               0

In [5]:
df['protocol_encoded'] = protocol_encoder.fit_transform(df['protocol'])
feature_cols = ['packet_size', 'port', 'duration', 'protocol_encoded']
X = df[feature_cols]
y = df['malicious_flag']
print ('Features: %s' % str(feature_cols))
print ('Target distribution: %s' % str(y.value_counts().to_dict()))

Features: ['packet_size', 'port', 'duration', 'protocol_encoded']
Target distribution: {0: 1400, 1: 600}


In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print ('Train: %d, Test: %d' % (len(X_train), len(X_test)))

Train: 1500, Test: 500


<hr>
## **Logistic Regression**

In [7]:
lr = LogisticRegression(random_state=42, max_iter=1000)
lr.fit(X_train_scaled, y_train)
y_pred_lr = lr.predict(X_test_scaled)
acc_lr = accuracy_score(y_test, y_pred_lr)
print ('LogisticRegression Accuracy: %.4f' % acc_lr)
print ('LogisticRegression Confusion Matrix:')
print (confusion_matrix(y_test, y_pred_lr))

LogisticRegression Accuracy: 0.7040
LogisticRegression Confusion Matrix:
[[252 103]
 [ 45 100]]


<hr>
## **Random Forest**

In [8]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)  # no scaling needed for RF
y_pred_rf = rf.predict(X_test)
acc_rf = accuracy_score(y_test, y_pred_rf)
print ('RandomForest Accuracy: %.4f' % acc_rf)
print ('RandomForest Confusion Matrix:')
print (confusion_matrix(y_test, y_pred_rf))

RandomForest Accuracy: 0.7900
RandomForest Confusion Matrix:
[[305  50]
 [ 55  90]]


In [9]:
print ('\n%s' % classification_report(y_test, y_pred_rf))


              precision    recall  f1-score   support

           0       0.85      0.86      0.85       355
           1       0.64      0.62      0.63       145

    accuracy                           0.79       500
   macro avg       0.74      0.74      0.74       500
weighted avg       0.79      0.79      0.79       500


<hr>
## **Cross-Validation**

In [10]:
cv_scores_lr = cross_val_score(LogisticRegression(random_state=42, max_iter=1000), X_train_scaled, y_train, cv=5)
cv_scores_rf = cross_val_score(RandomForestClassifier(n_estimators=100, random_state=42), X_train, y_train, cv=5)
print ('LogisticRegression CV accuracy: %.4f (+/- %.4f)' % (cv_scores_lr.mean(), cv_scores_lr.std() * 2))
print ('RandomForest CV accuracy:      %.4f (+/- %.4f)' % (cv_scores_rf.mean(), cv_scores_rf.std() * 2))

LogisticRegression CV accuracy: 0.6987 (+/- 0.0345)
RandomForest CV accuracy:      0.7880 (+/- 0.0289)


<hr>
## **Feature Importance (RandomForest)**

In [11]:
importances = rf.feature_importances_
indices = np.argsort(importances)[::-1]
print ('Feature ranking:')
for i in range(len(feature_cols)):
    print ('  %d. %s (%.4f)' % (i + 1, feature_cols[indices[i]], importances[indices[i]]))

Feature ranking:
  1. duration (0.3456)
  2. packet_size (0.2876)
  3. port (0.2345)
  4. protocol_encoded (0.1323)


In [12]:
plt.figure(figsize=(8, 4))
plt.barh([feature_cols[i] for i in indices], importances[indices], color='coral')
plt.xlabel('Importance')
plt.title('**RandomForest Feature Importance**')
plt.tight_layout()
print ('Feature importance plot displayed\n')

Feature importance plot displayed


<hr>
## **Summary**
**RandomForest (79% acc)** outperforms **LogisticRegression (70.4% acc)** on this synthetic network data.
**duration** and **packet_size** are the most informative features for detecting malicious traffic.

In [13]:
print ('Network intrusion prediction experiment complete.\n')

Network intrusion prediction experiment complete.
